# Лабораторная работа 11
## Лебедева Анна

1. Вариант легкий: Решите задачу классификации цветков ирисов с использованием PySpark

Установка

In [8]:
!pip install pyspark findspark -q

Запуск

In [12]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()
print("Spark запущен:", spark.version)

Spark запущен: 4.1.1


Импорты

In [10]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler

Загрузка данных

In [46]:
df = spark.read.csv("iris.csv", inferSchema=True, header=True)
df.show()
print("Всего строк:", df.count())

+------------+-----------+------------+-----------+-------+
|sepal.length|sepal.width|petal.length|petal.width|variety|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| Setosa|
|         4.9|        3.0|         1.4|        0.2| Setosa|
|         4.7|        3.2|         1.3|        0.2| Setosa|
|         4.6|        3.1|         1.5|        0.2| Setosa|
|         5.0|        3.6|         1.4|        0.2| Setosa|
|         5.4|        3.9|         1.7|        0.4| Setosa|
|         4.6|        3.4|         1.4|        0.3| Setosa|
|         5.0|        3.4|         1.5|        0.2| Setosa|
|         4.4|        2.9|         1.4|        0.2| Setosa|
|         4.9|        3.1|         1.5|        0.1| Setosa|
|         5.4|        3.7|         1.5|        0.2| Setosa|
|         4.8|        3.4|         1.6|        0.2| Setosa|
|         4.8|        3.0|         1.4|        0.1| Setosa|
|         4.3|        3.0|         1.1| 

Алгоритм не понимает слова, поэтому заменяем названия цветков на числа:
Setosa - 0, Versicolor - 1, Virginica - 2

In [13]:
indexer = StringIndexer(inputCol="variety", outputCol="label")
df_indexed = indexer.fit(df).transform(df)
df_indexed.show(5)

+------------+-----------+------------+-----------+-------+-----+
|sepal.length|sepal.width|petal.length|petal.width|variety|label|
+------------+-----------+------------+-----------+-------+-----+
|         5.1|        3.5|         1.4|        0.2| Setosa|  0.0|
|         4.9|        3.0|         1.4|        0.2| Setosa|  0.0|
|         4.7|        3.2|         1.3|        0.2| Setosa|  0.0|
|         4.6|        3.1|         1.5|        0.2| Setosa|  0.0|
|         5.0|        3.6|         1.4|        0.2| Setosa|  0.0|
+------------+-----------+------------+-----------+-------+-----+
only showing top 5 rows


LogisticRegression ожидает все признаки в одной колонке типа "вектор".
VectorAssembler склеивает 4 числовые колонки в один вектор [5.1, 3.5, 1.4, 0.2]
для каждой строки

In [14]:
assembler = VectorAssembler(
    inputCols=["`sepal.length`", "`sepal.width`", "`petal.length`", "`petal.width`"],
    outputCol="features"
)
df_assembled = assembler.transform(df_indexed)
df_assembled.show(5)

+------------+-----------+------------+-----------+-------+-----+-----------------+
|sepal.length|sepal.width|petal.length|petal.width|variety|label|         features|
+------------+-----------+------------+-----------+-------+-----+-----------------+
|         5.1|        3.5|         1.4|        0.2| Setosa|  0.0|[5.1,3.5,1.4,0.2]|
|         4.9|        3.0|         1.4|        0.2| Setosa|  0.0|[4.9,3.0,1.4,0.2]|
|         4.7|        3.2|         1.3|        0.2| Setosa|  0.0|[4.7,3.2,1.3,0.2]|
|         4.6|        3.1|         1.5|        0.2| Setosa|  0.0|[4.6,3.1,1.5,0.2]|
|         5.0|        3.6|         1.4|        0.2| Setosa|  0.0|[5.0,3.6,1.4,0.2]|
+------------+-----------+------------+-----------+-------+-----+-----------------+
only showing top 5 rows


Разбиваем данные: 80% на обучение, 20% на проверку

In [40]:
train, test = df_assembled.randomSplit([0.8, 0.2], seed=40)
print("Обучающая выборка:", train.count(), "строк")
print("Тестовая выборка:", test.count(), "строк")
print("Всего:", train.count() + test.count(), "строк")

Обучающая выборка: 120 строк
Тестовая выборка: 30 строк
Всего: 150 строк


Обучаем модель логистической регрессии на тренировочных данных.

In [51]:
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train)
print("Модель обучена")

Модель обучена


Применяем обученную модель к тестовой выборке — данным, которые
модель раньше не видела. Метод .transform() добавляет колонку
prediction с предсказанными классами

In [52]:
predictions = model.transform(test)
predictions.select("variety", "label", "prediction").show()

+----------+-----+----------+
|   variety|label|prediction|
+----------+-----+----------+
|    Setosa|  0.0|       0.0|
|    Setosa|  0.0|       0.0|
|    Setosa|  0.0|       0.0|
|    Setosa|  0.0|       0.0|
|    Setosa|  0.0|       0.0|
|Versicolor|  1.0|       1.0|
|    Setosa|  0.0|       0.0|
|    Setosa|  0.0|       0.0|
|    Setosa|  0.0|       0.0|
|Versicolor|  1.0|       1.0|
|Versicolor|  1.0|       1.0|
|    Setosa|  0.0|       0.0|
|    Setosa|  0.0|       0.0|
| Virginica|  2.0|       2.0|
|Versicolor|  1.0|       1.0|
|    Setosa|  0.0|       0.0|
|Versicolor|  1.0|       1.0|
|Versicolor|  1.0|       1.0|
|Versicolor|  1.0|       1.0|
|Versicolor|  1.0|       1.0|
+----------+-----+----------+
only showing top 20 rows


Считаем точность — долю правильных предсказаний

In [53]:
from pyspark.sql.functions import col

evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

total = predictions.count()
correct = predictions.filter(col("label") == col("prediction")).count()

print(f"Точность:      {evaluator.evaluate(predictions)*100:.1f}%")
print(f"Верно:         {correct}/{total}")
print(f"Ошибок:        {total - correct}")

Точность:      100.0%
Верно:         30/30
Ошибок:        0


2.	Решите задачу классификации пассажиров титаника с использованием PySpark [https://www.kaggle.com/c/titanic]

Загружаем датасет. Целевой признак — Survived (1 = выжил, 0 = не выжил)

In [55]:
df_titanic = spark.read.csv("train.csv", inferSchema=True, header=True)
df_titanic.show(5)
print("Всего строк:", df_titanic.count())

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

Проверяем пропуски в данных и решаем какие признаки использовать.

Берём:
- Pclass   — класс билета (1, 2, 3)
- Sex      — пол (переведём в числа)
- Age      — возраст (177 пропусков — заполним средним)
- SibSp    — кол-во братьев/сестёр/супругов на борту
- Parch    — кол-во родителей/детей на борту
- Fare     — цена билета

Не берём:
- PassengerId, Name, Ticket — не влияют на выживание
- Cabin    — 687 пропусков из 891, слишком много
- Embarked — 2 пропуска, удалим эти строки

In [56]:
from pyspark.sql.functions import col, sum as spark_sum

df_titanic.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_titanic.columns
]).show()

+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|PassengerId|Survived|Pclass|Name|Sex|Age|SibSp|Parch|Ticket|Fare|Cabin|Embarked|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|          0|       0|     0|   0|  0|177|    0|    0|     0|   0|  687|       2|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+



Обрабатываем пропуски

In [57]:
from pyspark.sql.functions import mean

# Считаем среднее значение возраста
avg_age = df_titanic.select(mean("Age")).first()[0]
print(f"Средний возраст: {avg_age:.1f}")

# Заполняем пропуски и выбираем нужные колонки
df_titanic = df_titanic.fillna({"Age": avg_age}) \
    .select("Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare")

df_titanic.show(5)
print("Пропуски:", df_titanic.filter(col("Age").isNull()).count())

Средний возраст: 29.7
+--------+------+------+----+-----+-----+-------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|   Fare|
+--------+------+------+----+-----+-----+-------+
|       0|     3|  male|22.0|    1|    0|   7.25|
|       1|     1|female|38.0|    1|    0|71.2833|
|       1|     3|female|26.0|    0|    0|  7.925|
|       1|     1|female|35.0|    1|    0|   53.1|
|       0|     3|  male|35.0|    0|    0|   8.05|
+--------+------+------+----+-----+-----+-------+
only showing top 5 rows
Пропуски: 0


Пол содержит текст (male/female), переводим в числа: female - 0, male - 1

In [58]:
indexer = StringIndexer(inputCol="Sex", outputCol="SexIndex")
df_titanic = indexer.fit(df_titanic).transform(df_titanic)
df_titanic.select("Sex", "SexIndex").show(5)

+------+--------+
|   Sex|SexIndex|
+------+--------+
|  male|     0.0|
|female|     1.0|
|female|     1.0|
|female|     1.0|
|  male|     0.0|
+------+--------+
only showing top 5 rows


Собираем все числовые признаки в один вектор для модели

In [59]:
assembler = VectorAssembler(
    inputCols=["Pclass", "SexIndex", "Age", "SibSp", "Parch", "Fare"],
    outputCol="features"
)
df_titanic = assembler.transform(df_titanic)
df_titanic.select("features", "Survived").show(5)

+--------------------+--------+
|            features|Survived|
+--------------------+--------+
|[3.0,0.0,22.0,1.0...|       0|
|[1.0,1.0,38.0,1.0...|       1|
|[3.0,1.0,26.0,0.0...|       1|
|[1.0,1.0,35.0,1.0...|       1|
|[3.0,0.0,35.0,0.0...|       0|
+--------------------+--------+
only showing top 5 rows


Разбиваем данные: 80% на обучение, 20% на проверку

In [71]:
train, test = df_titanic.randomSplit([0.8, 0.2], seed=45)
print("Обучающая выборка:", train.count(), "строк")
print("Тестовая выборка:", test.count(), "строк")

Обучающая выборка: 713 строк
Тестовая выборка: 178 строк


Обучаем модель логистической регрессии.
Целевой признак — Survived (1 = выжил, 0 = не выжил).

In [72]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="Survived")
model = lr.fit(train)
print("Модель обучена")

Модель обучена


Проверяем качество модели на тестовых данных

In [73]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col

predictions = model.transform(test)

evaluator = MulticlassClassificationEvaluator(
    labelCol="Survived",
    predictionCol="prediction",
    metricName="accuracy"
)

total = predictions.count()
correct = predictions.filter(col("Survived") == col("prediction")).count()

print(f"Точность:      {evaluator.evaluate(predictions)*100:.1f}%")
print(f"Верно:         {correct}/{total}")
print(f"Ошибок:        {total - correct}")

Точность:      75.3%
Верно:         134/178
Ошибок:        44
